# A data matrix, its row lines and its column lines

Given a matrix of numbers, this notebook draws two line charts:

* **one line per row**: the x axis is the columns
* **one line per column**: the x axis is the rows

Then it puts both charts in a small Dash app **inside the notebook**, and shows the two ways the charts refresh
when the data changes:

1. you edit a cell of the table in the app
2. you change the matrix from Python, in a notebook cell

All the numbers are random, made up in the notebook.

**Before you start:** `pip install -r requirements.txt` from the repo root (it includes `jupyterlab` and
`dash-ag-grid`), then open this file with `jupyter lab` or in VS Code. Run the cells in order with Shift+Enter.

In [ ]:
import numpy as np
import pandas as pd
import plotly.express as px

ROWS = [f"row {i}" for i in range(1, 6)]
COLS = [f"c{j}" for j in range(1, 9)]


def random_matrix(seed):
    """A made-up 5 x 8 matrix: each row is a smooth curve plus a little noise."""
    rng = np.random.default_rng(seed)
    x = np.linspace(0, 1, len(COLS))
    values = [rng.uniform(1, 5) + rng.uniform(-2, 2) * x + rng.normal(0, 0.2, len(COLS)) for _ in ROWS]
    return pd.DataFrame(np.round(values, 2), index=pd.Index(ROWS, name="row"), columns=pd.Index(COLS, name="column"))


M = random_matrix(0)
M

## Step A: the two charts with Plotly alone

`px.line(df)` draws **one line per column** of `df`, with the index on the x axis. So:

* `px.line(M)` gives one line per column of `M` (x = the rows)
* `px.line(M.T)` gives one line per row of `M`, because `M.T` (the transpose) turns rows into columns

Naming the index and the columns (`row`, `column`) gives the axes and the legend their titles.

In [ ]:
def row_lines(df):
    return px.line(df.T, markers=True, title="One line per row")


def col_lines(df):
    return px.line(df, markers=True, title="One line per column")


row_lines(M).show()
col_lines(M).show()

These charts are **snapshots**. Change `M` now (say `M.loc["row 1"] = 0`) and they stay as they are; you have to
re-run the cell above. To make charts that follow the data, the next step puts them in a Dash app.

## Step B: a Dash app in the notebook

The app has a table (an AG Grid you can edit) and the two charts side by side. The data lives in one place,
`live`, a small object holding the current matrix and a version number:

* **an edit in the table** fires `draw`, which redraws both charts and copies the edited matrix into `live.df`
  (so the notebook can read your edits)
* **`live.set(new_matrix)`** in a notebook cell stores the new matrix and adds 1 to `live.version`

How does the page find out about `live.set`? It asks. A `dcc.Interval` fires twice a second and runs `pull`, which
compares `live.version` with the version the page last saw (kept in a `dcc.Store`). If they differ, it sends the
new matrix to the table, and the new table data fires `draw`. If they are the same, `PreventUpdate` leaves
the page alone. A web page can only ask the server; the server cannot push to it, which is why it polls.

`app.run(jupyter_mode="inline")` shows the app under the cell, and the cell finishes at once: the app's server keeps
running in the background while you run other cells. Use `jupyter_mode="tab"` to open it in a browser tab instead.

In [ ]:
import dash_ag_grid as dag
from dash import Dash, html, dcc, Input, Output, State
from dash.exceptions import PreventUpdate


class LiveMatrix:
    """The current matrix. Notebook cells change it with set(); the app checks the version twice a second."""

    def __init__(self, df):
        self.df = df.copy()
        self.version = 0

    def set(self, df):
        self.df = df.copy()
        self.version += 1


live = LiveMatrix(M)


def to_rows(df):
    """DataFrame -> the grid's rowData (a list of dicts, one per row)."""
    return df.reset_index().to_dict("records")


def to_frame(rows):
    """The grid's rowData -> DataFrame with the row labels as the index."""
    df = pd.DataFrame(rows).set_index("row")[COLS].apply(pd.to_numeric, errors="coerce")
    df.columns.name = "column"
    return df


app = Dash(__name__)
app.layout = html.Div([
    dag.AgGrid(
        id="grid",
        rowData=to_rows(live.df),
        columnDefs=[{"field": "row", "headerName": "row", "editable": False, "width": 90}]
                   + [{"field": c, "headerName": c, "editable": True, "width": 80} for c in COLS],
        dashGridOptions={"domLayout": "autoHeight"},
        style={"width": 760},
    ),
    html.Div([dcc.Graph(id="by-row", style={"flex": 1}), dcc.Graph(id="by-col", style={"flex": 1})],
             style={"display": "flex"}),
    dcc.Interval(id="tick", interval=500),       # every 500 ms
    dcc.Store(id="seen", data=live.version),     # the version this page shows
])


@app.callback(Output("grid", "rowData"),
              Output("seen", "data"),
              Input("tick", "n_intervals"),
              State("seen", "data"))
def pull(_, seen):
    if live.version == seen:
        raise PreventUpdate                      # nothing new: leave the page alone
    return to_rows(live.df), live.version


@app.callback(Output("by-row", "figure"),
              Output("by-col", "figure"),
              Input("grid", "rowData"),           # a new matrix from pull
              Input("grid", "cellValueChanged"))  # an edit in the table
def draw(rows, edit):
    df = to_frame(rows)
    live.df = df                                 # keep edits, so the notebook can read them
    return row_lines(df), col_lines(df)


app.run(jupyter_mode="inline", port=8060, jupyter_height=820)

**Try it:** double-click a cell in the table, type a number, press Enter. Both charts redraw: one point moves
in the row chart, and the same value moves in the column chart.

## Step C: change the data from Python

Run the cells below one at a time and watch the app above. Each refreshes it within half a second.

In [ ]:
new = live.df.copy()
new.loc["row 3"] = new.loc["row 3"] + 2      # lift every value of row 3 by 2
live.set(new)

In [ ]:
new = live.df.copy()
new["c8"] = 0.0                              # set the whole last column to 0
live.set(new)

In [ ]:
live.set(random_matrix(seed=42))             # a completely new matrix

Edits made in the table come back to Python too. Edit a cell in the app, then run:

In [ ]:
live.df

## How it fits together

```
notebook cell ── live.set(df) ──▶ live (df, version)
                                     ▲          │
                         draw copies │          │ pull, every 500 ms,
                          edits back │          ▼ if the version changed
          table edit ──▶ grid rowData ◀─────────┘
                              │
                              ▼ draw
                   row chart + column chart
```

* `pull` and `draw` are ordinary Dash callbacks, as in steps 3 to 7 of the tutorial.
* The same pattern works for any data that changes outside the app (a file, a database table): replace
  `live.version` with a cheap check that tells you something changed, such as a file's modification time or
  the largest id in a table.
* The app keeps running until the kernel stops. To change the app's code, edit the cell and run it again;
  it restarts on the same port.

**Things to try**

1. Make the matrix 10 x 20 (change `ROWS` and `COLS`, then run everything again).
2. Add a third chart: a heatmap of the matrix, `px.imshow(df)`, as one more Output of `draw`.
3. Slow the polling to `interval=2000` and see the delay after `live.set`.